In [1]:
!pip install unstructured pymupdf pandas
!pip install "unstructured[pdf]"
!pip install layoutparser pillow pdfminer.six

In [2]:
# RefAIned Unstructured Loader Benchmark

#This notebook benchmarks document parsing approaches for RefAIned, with emphasis on **Unstructured** for academic PDFs.

## Targets
## - Preserve title, authors, sections, references, and tables
## - Improve chunk quality over raw text extraction
## - Produce structured outputs for retrieval and future graph storage

## Planned flow
## 1. Load a small set of academic PDFs
## 2. Run a **PyMuPDF baseline**
## 3. Run an **Unstructured parser**
## 4. Compare outputs side by side
## 5. Normalize parsed results into a shared schema
## 6. Test chunking strategies
## 7. Prepare chunks for embeddings and retrieval

In [3]:
# Core imports
from pathlib import Path
import json
import time
import re
from collections import Counter, defaultdict

import pandas as pd

# Baseline parser
import fitz  # PyMuPDF

# Unstructured parser
from unstructured.partition.pdf import partition_pdf

# Optional pretty printing
from pprint import pprint

# Display settings
pd.set_option("display.max_colwidth", 200)

print("Imports loaded successfully.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

Imports loaded successfully.


In [4]:
# Correct path to your PDFs
PDF_DIR = Path("CSE195/data/test_papers")

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print(f"PDF directory: {PDF_DIR.resolve()}")
print(f"Found {len(pdf_files)} PDF file(s):")
for f in pdf_files:
    print("-", f.name)

PDF directory: /home/jovyan/CSE195/data/test_papers
Found 1 PDF file(s):
- 2410.21169v4.pdf


In [5]:
## 2. Helper functions

##These helpers will:
## 1. extract baseline text with PyMuPDF
## 2. extract semantic elements with Unstructured
## 3. preview outputs
## 4. score basic structure quality

In [6]:
def extract_pymupdf_text(pdf_path: Path) -> dict:
    """
    Baseline extraction using PyMuPDF.
    Returns flat text plus page-level information.
    """
    start = time.time()
    doc = fitz.open(pdf_path)
    
    pages = []
    full_text_parts = []
    
    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page_num": i + 1,
            "text": text
        })
        full_text_parts.append(text)
    
    elapsed = time.time() - start
    full_text = "\n".join(full_text_parts)
    
    return {
        "parser": "pymupdf",
        "file_name": pdf_path.name,
        "num_pages": len(doc),
        "text": full_text,
        "pages": pages,
        "elapsed_seconds": round(elapsed, 4)
    }


def extract_unstructured_elements(pdf_path: Path) -> dict:
    """
    Structured extraction using Unstructured.
    Returns semantic elements and simple metadata.
    """
    start = time.time()
    
    elements = partition_pdf(
        filename=str(pdf_path),
        strategy="fast",  # good starting point; later you can compare with hi_res
        infer_table_structure=True,
        include_metadata=True,
    )
    
    elapsed = time.time() - start
    
    structured = []
    for idx, el in enumerate(elements):
        text = str(el).strip()
        category = getattr(el, "category", type(el).__name__)
        metadata = getattr(el, "metadata", None)
        
        structured.append({
            "element_index": idx,
            "type": category,
            "text": text,
            "metadata": str(metadata) if metadata else None
        })
    
    return {
        "parser": "unstructured",
        "file_name": pdf_path.name,
        "num_elements": len(structured),
        "elements": structured,
        "elapsed_seconds": round(elapsed, 4)
    }


def preview_text(text: str, n: int = 1200) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text[:n]


def preview_elements(elements: list, max_items: int = 10) -> pd.DataFrame:
    rows = []
    for el in elements[:max_items]:
        rows.append({
            "idx": el["element_index"],
            "type": el["type"],
            "text_preview": preview_text(el["text"], 180)
        })
    return pd.DataFrame(rows)


def count_element_types(elements: list) -> pd.DataFrame:
    counts = Counter(el["type"] for el in elements)
    df = pd.DataFrame(
        [{"element_type": k, "count": v} for k, v in counts.items()]
    ).sort_values("count", ascending=False)
    return df.reset_index(drop=True)

In [7]:
test_pdf = pdf_files[0]
result = extract_pymupdf_text(test_pdf)

print("Parser:", result["parser"])
print("File:", result["file_name"])
print("Pages:", result["num_pages"])
print("Elapsed:", result["elapsed_seconds"], "seconds")
print()
print(preview_text(result["text"], 1000))

Parser: pymupdf
File: 2410.21169v4.pdf
Pages: 42
Elapsed: 0.1964 seconds

Document Parsing Unveiled: Techniques, Challenges, and Prospects for Structured Data Extraction QINTONG ZHANG∗∗, Peking University, China BIN WANG∗, Shanghai Artificial Intelligence Laboratory, China VICTOR SHEA-JAY HUANG, Beihang University, China JUNYUAN ZHANG, Shanghai Artificial Intelligence Laboratory, China ZHENGREN WANG, Peking University, China HAO LIANG, Peking University, China CONGHUI HE†, Shanghai Artificial Intelligence Laboratory, China WENTAO ZHANG, Peking University, China Document parsing is essential for converting unstructured and semi-structured documents—such as contracts, academic papers, and invoices—into structured, machine-readable data. Document parsing reliable structured data from unstructured inputs, providing huge convenience for numerous applications. Especially with recent achievements in Large Language Models, document parsing plays an indispensable role in both knowledge base con

In [8]:
## 3. Run baseline PyMuPDF extraction

In [9]:
pymupdf_results = []

for pdf_path in pdf_files:
    result = extract_pymupdf_text(pdf_path)
    pymupdf_results.append(result)

print(f"Completed PyMuPDF extraction for {len(pymupdf_results)} file(s).")

Completed PyMuPDF extraction for 1 file(s).


In [10]:
# Preview baseline extraction for the first PDF
if pymupdf_results:
    first = pymupdf_results[0]
    print("File:", first["file_name"])
    print("Pages:", first["num_pages"])
    print("Elapsed seconds:", first["elapsed_seconds"])
    print()
    print(preview_text(first["text"], 2000))
else:
    print("No PDF results available.")

File: 2410.21169v4.pdf
Pages: 42
Elapsed seconds: 0.1562

Document Parsing Unveiled: Techniques, Challenges, and Prospects for Structured Data Extraction QINTONG ZHANG∗∗, Peking University, China BIN WANG∗, Shanghai Artificial Intelligence Laboratory, China VICTOR SHEA-JAY HUANG, Beihang University, China JUNYUAN ZHANG, Shanghai Artificial Intelligence Laboratory, China ZHENGREN WANG, Peking University, China HAO LIANG, Peking University, China CONGHUI HE†, Shanghai Artificial Intelligence Laboratory, China WENTAO ZHANG, Peking University, China Document parsing is essential for converting unstructured and semi-structured documents—such as contracts, academic papers, and invoices—into structured, machine-readable data. Document parsing reliable structured data from unstructured inputs, providing huge convenience for numerous applications. Especially with recent achievements in Large Language Models, document parsing plays an indispensable role in both knowledge base construction and tr

In [11]:
## 4. Run Unstructured extraction

In [12]:
unstructured_results = []

for pdf_path in pdf_files:
    result = extract_unstructured_elements(pdf_path)
    unstructured_results.append(result)

print(f"Completed Unstructured extraction for {len(unstructured_results)} file(s).")

Completed Unstructured extraction for 1 file(s).


In [13]:
# Preview structured extraction for the first PDF
if unstructured_results:
    first_u = unstructured_results[0]
    print("File:", first_u["file_name"])
    print("Elements:", first_u["num_elements"])
    print("Elapsed seconds:", first_u["elapsed_seconds"])
    print()
    display(preview_elements(first_u["elements"], max_items=15))
else:
    print("No Unstructured results available.")

File: 2410.21169v4.pdf
Elements: 1950
Elapsed seconds: 36.5739



,idx,type,text_preview
0,0,UncategorizedText,5 2 0 2
1,1,Title,r p A 6 1
2,2,UncategorizedText,]
3,3,Title,M M
4,4,Title,. s c [
5,5,UncategorizedText,4 v 9 6 1 1 2 . 0 1 4 2 : v i X r a
6,6,UncategorizedText,"Document Parsing Unveiled: Techniques, Challenges, and Prospects for Structured Data Extraction QINTONG ZHANG∗∗, Peking University, China BIN WANG∗, Shanghai Artificial Intelligenc"
7,7,NarrativeText,"Document parsing is essential for converting unstructured and semi-structured documents—such as contracts, academic papers, and"
8,8,NarrativeText,"invoices—into structured, machine-readable data. Document parsing reliable structured data from unstructured inputs, providing"
9,9,NarrativeText,"huge convenience for numerous applications. Especially with recent achievements in Large Language Models, document parsing"


In [14]:
# Element type distribution for first PDF
if unstructured_results:
    first_u = unstructured_results[0]
    display(count_element_types(first_u["elements"]))
else:
    print("No Unstructured results available.")

,element_type,count
0,NarrativeText,1101
1,Title,541
2,UncategorizedText,277
3,ListItem,31


In [15]:
## 5. Basic quality scoring

## This is a lightweight manual/heuristic comparison table.

## Scoring:
## - 0 = failed
## - 1 = partial
## - 2 = good

In [16]:
def heuristic_score_pymupdf(text: str) -> dict:
    text_lower = text.lower()
    
    title_quality = 1 if len(text.strip()) > 0 else 0
    author_quality = 1 if "abstract" in text_lower[:3000] else 0
    section_quality = 2 if any(x in text_lower for x in ["introduction", "method", "results", "conclusion"]) else 1
    references_quality = 2 if "references" in text_lower else 0
    table_quality = 0  # flat extraction usually poor for tables
    
    return {
        "title_quality": title_quality,
        "author_quality": author_quality,
        "section_quality": section_quality,
        "references_quality": references_quality,
        "table_quality": table_quality
    }


def heuristic_score_unstructured(elements: list) -> dict:
    types = [e["type"].lower() for e in elements]
    text_blob = " ".join(e["text"] for e in elements).lower()
    
    title_quality = 2 if any("title" in t for t in types) else 1
    author_quality = 1  # depends on parser output and paper style
    section_quality = 2 if any("title" in t or "header" in t for t in types) else 1
    references_quality = 2 if "references" in text_blob else 1
    table_quality = 2 if any("table" in t for t in types) else 0
    
    return {
        "title_quality": title_quality,
        "author_quality": author_quality,
        "section_quality": section_quality,
        "references_quality": references_quality,
        "table_quality": table_quality
    }

In [17]:
comparison_rows = []

for py_res in pymupdf_results:
    scores = heuristic_score_pymupdf(py_res["text"])
    comparison_rows.append({
        "file_name": py_res["file_name"],
        "parser": "pymupdf",
        "elapsed_seconds": py_res["elapsed_seconds"],
        **scores
    })

for un_res in unstructured_results:
    scores = heuristic_score_unstructured(un_res["elements"])
    comparison_rows.append({
        "file_name": un_res["file_name"],
        "parser": "unstructured",
        "elapsed_seconds": un_res["elapsed_seconds"],
        **scores
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(["file_name", "parser"]).reset_index(drop=True)
display(comparison_df)

,file_name,parser,elapsed_seconds,title_quality,author_quality,section_quality,references_quality,table_quality
0,2410.21169v4.pdf,pymupdf,0.1562,1,0,2,2,0
1,2410.21169v4.pdf,unstructured,36.5739,2,1,2,2,0


In [18]:
## 6. Normalize Unstructured output into a RefAIned schema

## This creates a first-pass scholarly document schema:
## - title
## - authors
## - abstract
## - sections
## - references
## - tables

In [19]:
def normalize_unstructured_result(un_res: dict) -> dict:
    elements = un_res["elements"]
    
    title = None
    abstract_parts = []
    sections = []
    references = []
    tables = []
    
    current_section = {
        "section_title": "Front Matter",
        "content": [],
        "element_types": []
    }
    
    for el in elements:
        el_type = el["type"]
        el_text = el["text"].strip()
        
        if not el_text:
            continue
        
        if title is None and "Title" in el_type:
            title = el_text
        
        # crude abstract detection
        if el_text.lower().startswith("abstract"):
            abstract_parts.append(el_text)
        
        # simple section handling
        if "Title" in el_type and title is not None and el_text != title:
            if current_section["content"]:
                sections.append({
                    "section_title": current_section["section_title"],
                    "content": "\n".join(current_section["content"]),
                    "element_types": current_section["element_types"]
                })
            current_section = {
                "section_title": el_text,
                "content": [],
                "element_types": []
            }
            continue
        
        if "Table" in el_type:
            tables.append(el_text)
        
        if "reference" in el_text.lower():
            references.append(el_text)
        
        current_section["content"].append(el_text)
        current_section["element_types"].append(el_type)
    
    if current_section["content"]:
        sections.append({
            "section_title": current_section["section_title"],
            "content": "\n".join(current_section["content"]),
            "element_types": current_section["element_types"]
        })
    
    normalized = {
        "doc_id": un_res["file_name"].replace(".pdf", ""),
        "source_file": un_res["file_name"],
        "title": title,
        "authors": [],
        "abstract": "\n".join(abstract_parts).strip(),
        "sections": sections,
        "tables": tables,
        "references": references,
        "metadata": {
            "parser": "unstructured"
        }
    }
    
    return normalized

In [20]:
normalized_docs = [normalize_unstructured_result(r) for r in unstructured_results]

print(f"Normalized {len(normalized_docs)} document(s).")
if normalized_docs:
    pprint(normalized_docs[0])

Normalized 1 document(s).
{'abstract': '',
 'authors': [],
 'doc_id': '2410.21169v4',
 'metadata': {'parser': 'unstructured'},
 'references': ['[180] Ihsin Tsaiyun Phillips. 1996. User’s reference manual '
                'for the UW english/technical document image database III. '
                'UW-III English/technical document',
                'reference text. Detailed metrics for OCR tasks are listed in '
                'Table 8.'],
 'sections': [{'content': '5 2 0 2\nr p A 6 1\n]',
               'element_types': ['UncategorizedText',
                                 'Title',
                                 'UncategorizedText'],
               'section_title': 'Front Matter'},
              {'content': '4 v 9 6 1 1 2 . 0 1 4 2 : v i X r a\n'
                          'Document Parsing Unveiled: Techniques, Challenges, '
                          'and Prospects for Structured Data Extraction '
                          'QINTONG ZHANG∗∗, Peking University, China BIN '
         

In [21]:
import re
from pathlib import Path

def clean_text(s):
    if s is None:
        return ""
    s = s.replace("\u00ad", "")          # soft hyphen
    s = s.replace("\ufb01", "fi")        # ligatures if present
    s = s.replace("\ufb02", "fl")
    s = s.replace("￾", "")
    s = s.replace("\xa0", " ")
    s = re.sub(r"-\n(?=[a-z])", "", s)   # de-hyphenate line breaks
    s = re.sub(r"\s+", " ", s).strip()
    return s

def is_junky_title(s):
    s = clean_text(s)
    if not s:
        return True
    if len(s) < 4:
        return True
    if len(re.findall(r"[A-Za-z]", s)) < 3:
        return True
    if re.fullmatch(r"[^\w]*", s):
        return True
    if re.search(r"\bbbox\b|\bx0\b|\by0\b|\bx1\b|\by1\b", s, re.I):
        return True
    if re.search(r"^[A-Za-z]\s+[A-Za-z]\s+[A-Za-z]", s):   # catches stuff like "r p A 6 1"
        return True
    return False

def looks_like_author_block(s):
    s = clean_text(s)
    if not s:
        return False
    has_email = "@" in s
    commas = s.count(",")
    semicolons = s.count(";")
    return has_email or commas >= 4 or semicolons >= 2

def extract_title_from_pdf_text(pdf_text):
    lines = [clean_text(x) for x in pdf_text.splitlines()]
    lines = [x for x in lines if x]

    bad_prefixes = (
        "manuscript submitted",
        "arxiv:",
        "authors’ addresses:",
        "authors' addresses:",
        "permission to make",
        "© ",
        "copyright",
    )

    candidates = []
    for line in lines[:60]:
        low = line.lower()
        if any(low.startswith(p) for p in bad_prefixes):
            continue
        if len(line) < 20 or len(line) > 220:
            continue
        if "@" in line:
            continue
        if re.match(r"^\d+(\.\d+)?\s+[A-Z]", line):
            continue
        candidates.append(line)

    title_like = []
    for c in candidates:
        words = c.split()
        cap_ratio = sum(1 for w in words if w[:1].isupper()) / max(len(words), 1)
        if len(words) >= 4 and cap_ratio >= 0.45:
            title_like.append(c)

    if title_like:
        title_like.sort(key=lambda x: (-len(x.split()), len(x)))
        return title_like[0]

    return candidates[0] if candidates else ""

def extract_authors_from_pdf_text(pdf_text):
    lines = [clean_text(x) for x in pdf_text.splitlines()]
    lines = [x for x in lines if x]

    author_lines = []
    for line in lines[:120]:
        if looks_like_author_block(line):
            author_lines.append(line)

    if not author_lines:
        return []

    joined = " ".join(author_lines)
    joined = re.sub(r"Authors[’'] addresses:\s*", "", joined, flags=re.I)

    emails = re.findall(r"[\w\.-]+@[\w\.-]+\.\w+", joined)
    names_part = joined
    for e in emails:
        names_part = names_part.replace(e, "")

    chunks = re.split(r";|,|\band\b", names_part)
    names = []
    for c in chunks:
        c = clean_text(c)
        if not c:
            continue
        if len(c.split()) < 2 or len(c.split()) > 6:
            continue
        if re.search(r"university|laboratory|china|beijing|shanghai|author|corresponding", c, re.I):
            continue
        if re.search(r"\d", c):
            continue
        names.append(c)

    deduped = []
    seen = set()
    for n in names:
        key = n.lower()
        if key not in seen:
            seen.add(key)
            deduped.append(n)

    return deduped

def extract_abstract_from_pdf_text(pdf_text):
    text = pdf_text

    # strongest case: explicit Abstract heading
    m = re.search(
        r"\bAbstract\b[:\s]*(.+?)(?=\n\s*(?:1\s+INTRODUCTION|1\s+Introduction|INTRODUCTION)\b)",
        text,
        flags=re.S
    )
    if m:
        return clean_text(m.group(1))

    # fallback: grab text before introduction, but after title/front matter noise
    m2 = re.search(
        r"(Document Parsing Unveiled.*?)(?=\n\s*(?:1\s+INTRODUCTION|1\s+Introduction|INTRODUCTION)\b)",
        text,
        flags=re.S
    )
    if m2:
        block = clean_text(m2.group(1))
        block = re.sub(r"^.*?Document Parsing Unveiled[^.]*\.\s*", "", block)
        return block[:2500]

    return ""

def repair_sections(sections):
    repaired = []
    buffer_content = []

    for sec in sections:
        title = clean_text(sec.get("section_title", ""))
        content = clean_text(sec.get("content", ""))
        element_types = sec.get("element_types", [])

        if is_junky_title(title):
            if content:
                buffer_content.append(content)
            continue

        if buffer_content:
            content = clean_text(" ".join(buffer_content) + " " + content)
            buffer_content = []

        repaired.append({
            "section_title": title,
            "content": content,
            "element_types": element_types,
        })

    if buffer_content and repaired:
        repaired[-1]["content"] = clean_text(repaired[-1]["content"] + " " + " ".join(buffer_content))

    return repaired

def repair_doc(doc, raw_pdf_text=None):
    doc = dict(doc)

    # clean existing fields
    doc["title"] = clean_text(doc.get("title", ""))
    doc["abstract"] = clean_text(doc.get("abstract", ""))
    doc["authors"] = doc.get("authors", []) or []
    doc["sections"] = repair_sections(doc.get("sections", []))

    # if raw PDF text is available, repair front matter
    if raw_pdf_text:
        repaired_title = extract_title_from_pdf_text(raw_pdf_text)
        repaired_authors = extract_authors_from_pdf_text(raw_pdf_text)
        repaired_abstract = extract_abstract_from_pdf_text(raw_pdf_text)

        if not doc["title"] or is_junky_title(doc["title"]):
            doc["title"] = repaired_title

        if not doc["authors"]:
            doc["authors"] = repaired_authors

        if not doc["abstract"]:
            doc["abstract"] = repaired_abstract

    # last fallback for title from sections
    if (not doc["title"] or is_junky_title(doc["title"])) and doc["sections"]:
        for sec in doc["sections"]:
            st = clean_text(sec.get("section_title", ""))
            if len(st) > 20 and not is_junky_title(st):
                doc["title"] = st
                break

    return doc

In [22]:
# Build raw PDF text with PyMuPDF for front-matter repair
raw_pdf_text_map = {}

for pdf_path in pdf_files:
    text_parts = []
    pdf_doc = fitz.open(pdf_path)
    for page in pdf_doc:
        text_parts.append(page.get_text("text"))
    pdf_doc.close()
    raw_pdf_text_map[pdf_path.stem] = "\n".join(text_parts)

# Repair all normalized docs
repaired_docs = []
for doc in normalized_docs:
    raw_text = raw_pdf_text_map.get(doc["doc_id"], "")
    repaired = repair_doc(doc, raw_text)
    repaired_docs.append(repaired)

# Quick preview
for doc in repaired_docs:
    print("=" * 80)
    print("DOC ID   :", doc["doc_id"])
    print("TITLE    :", doc.get("title", ""))
    print("AUTHORS  :", doc.get("authors", [])[:8])
    print("ABSTRACT :", doc.get("abstract", "")[:500], "...")
    print("SECTIONS :", len(doc.get("sections", [])))
    if doc.get("sections"):
        print("FIRST 5 SECTION TITLES:")
        for sec in doc["sections"][:5]:
            print("-", sec["section_title"])

DOC ID   : 2410.21169v4
TITLE    : Qintong Zhang, Bin Wang, Victor Shea-Jay Huang, Junyuan Zhang, Zhengren Wang, Hao Liang, Conghui He, and Wentao Zhang.
AUTHORS  : ['Qintong Zhang', 'Bin Wang', 'Victor Shea-Jay Huang', 'Junyuan Zhang', 'Zhengren Wang', 'Hao Liang', 'Conghui He', 'heconghui@pjlab. org.cn']
ABSTRACT : Document parsing reliable structured data from unstructured inputs, providing huge convenience for numerous applications. Especially with recent achievements in Large Language Models, document parsing plays an indispensable role in both knowledge base construction and training data generation. This survey presents a comprehensive review of the current state of document parsing, covering key methodologies, from modular pipeline systems to end-to-end models driven by large vision-language models.  ...
SECTIONS : 311
FIRST 5 SECTION TITLES:
- Front Matter
- CCS Concepts: • Computing methodologies → Natural language processing; Computer vision.
- ACM Reference Format:
- 42 pag

In [23]:
OUTPUT_DIR = Path("CSE195/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for doc in repaired_docs:
    out_path = OUTPUT_DIR / f"{doc['doc_id']}_repaired.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(doc, f, indent=2, ensure_ascii=False)

print(f"Saved repaired JSON files to: {OUTPUT_DIR.resolve()}")

Saved repaired JSON files to: /home/jovyan/CSE195/outputs


In [24]:
## 7. Save normalized outputs to JSON

In [25]:
# Save outputs inside the project directory
OUTPUT_DIR = Path("CSE195/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for doc in normalized_docs:
    out_path = OUTPUT_DIR / f"{doc['doc_id']}_normalized.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(doc, f, indent=2, ensure_ascii=False)

print(f"Saved normalized JSON files to: {OUTPUT_DIR.resolve()}")

Saved normalized JSON files to: /home/jovyan/CSE195/outputs


In [26]:
## 8. Chunking experiments

## We will compare:
## 1. naive fixed-size chunking
## 2. paragraph-style chunking
## 3. structure-aware chunking using normalized sections

In [27]:
def chunk_fixed(text: str, chunk_size: int = 1000, overlap: int = 150) -> list:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]


def chunk_paragraphs(text: str, max_chars: int = 1200) -> list:
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = []
    current_len = 0
    
    for p in paragraphs:
        if current_len + len(p) > max_chars and current:
            chunks.append("\n\n".join(current))
            current = [p]
            current_len = len(p)
        else:
            current.append(p)
            current_len += len(p)
    
    if current:
        chunks.append("\n\n".join(current))
    
    return chunks


def chunk_structured(normalized_doc: dict, max_chars: int = 1200) -> list:
    chunks = []
    
    if normalized_doc.get("abstract"):
        chunks.append({
            "section": "Abstract",
            "text": normalized_doc["abstract"]
        })
    
    for sec in normalized_doc.get("sections", []):
        sec_title = sec["section_title"]
        sec_text = sec["content"]
        
        if len(sec_text) <= max_chars:
            chunks.append({
                "section": sec_title,
                "text": sec_text
            })
        else:
            subchunks = chunk_paragraphs(sec_text, max_chars=max_chars)
            for sub in subchunks:
                chunks.append({
                    "section": sec_title,
                    "text": sub
                })
    
    return chunks

In [28]:
chunk_eval_rows = []

for py_res in pymupdf_results:
    fixed_chunks = chunk_fixed(py_res["text"])
    para_chunks = chunk_paragraphs(py_res["text"])
    
    chunk_eval_rows.append({
        "file_name": py_res["file_name"],
        "strategy": "fixed",
        "num_chunks": len(fixed_chunks),
        "avg_chunk_len": round(sum(len(c) for c in fixed_chunks) / max(len(fixed_chunks), 1), 2)
    })
    
    chunk_eval_rows.append({
        "file_name": py_res["file_name"],
        "strategy": "paragraph",
        "num_chunks": len(para_chunks),
        "avg_chunk_len": round(sum(len(c) for c in para_chunks) / max(len(para_chunks), 1), 2)
    })

for doc in normalized_docs:
    structured_chunks = chunk_structured(doc)
    chunk_eval_rows.append({
        "file_name": doc["source_file"],
        "strategy": "structured",
        "num_chunks": len(structured_chunks),
        "avg_chunk_len": round(
            sum(len(c["text"]) for c in structured_chunks) / max(len(structured_chunks), 1), 2
        )
    })

chunk_eval_df = pd.DataFrame(chunk_eval_rows).sort_values(["file_name", "strategy"]).reset_index(drop=True)
display(chunk_eval_df)

,file_name,strategy,num_chunks,avg_chunk_len
0,2410.21169v4.pdf,fixed,213,997.95
1,2410.21169v4.pdf,paragraph,42,4303.69
2,2410.21169v4.pdf,structured,322,498.78


In [29]:
# Preview chunk samples for the first normalized document
if normalized_docs:
    first_doc = normalized_docs[0]
    structured_chunks = chunk_structured(first_doc)
    
    preview_rows = []
    for i, ch in enumerate(structured_chunks[:10]):
        preview_rows.append({
            "chunk_id": i,
            "section": ch["section"],
            "text_preview": preview_text(ch["text"], 220)
        })
    
    display(pd.DataFrame(preview_rows))
else:
    print("No normalized documents available.")

,chunk_id,section,text_preview
0,0,Front Matter,5 2 0 2 r p A 6 1 ]
1,1,. s c [,"4 v 9 6 1 1 2 . 0 1 4 2 : v i X r a Document Parsing Unveiled: Techniques, Challenges, and Prospects for Structured Data Extraction QINTONG ZHANG∗∗, Peking University, China BIN WANG∗, Shanghai Ar..."
2,2,CCS Concepts: • Computing methodologies → Natural language processing; Computer vision.,"Additional Key Words and Phrases: Document Parsing, Document OCR, Document Layout Analysis, Vision-language Model"
3,3,ACM Reference Format:,"Qintong Zhang, Bin Wang, Victor Shea-Jay Huang, Junyuan Zhang, Zhengren Wang, Hao Liang, Conghui He, and Wentao Zhang. 2024. Document Parsing Unveiled: Techniques, Challenges, and Prospects for St..."
4,4,42 pages. https://doi.org/XXXXXXX.XXXXXXX,∗Authors contributed equally to this research.The work is done during an internship at Shanghai Artifcial Intelligence Laboratory. †Conghui He is the corresponding author. Authors’ addresses: Qint...
5,5,1 INTRODUCTION,"As digital transformation accelerates, electronic documents have increasingly replaced paper as the primary medium for information exchange across various industries. This shift has broadened the ..."
6,6,integration into modern workflows [68].,"Document parsing is crucial for document-related tasks, reshaping how information is stored, shared, and applied across numerous applications. It underpins various downstream processes, including ..."
7,7,Based on Vision Feature,"DiT [115] [137],DocGCN [151],GLAM [242]) BERTGrid [51], [41],DocLayout-YOLO [303]"
8,8,Integrate with Semantics,"LayoutLM [268],LayoutLMv2 [269],LayoutLMv3 [89] VSR[289],Unidoc [70], [255], [184],LayoutLLM [149]"
9,9,Text Detection,"Textboxes [126, 127], CTPN [229], DRRG [292], [312], DeepText [309], [42] [96], [155], [273], [140], [48, 262] PAN [249], CRAFT [12], SPCNET [263] LSAE [230], [114], EAST [310] CentripetalText [20..."


In [30]:
## 9. Prepare chunks for embeddings

## This cell creates a final chunk table with:
## - chunk id
## - source file
## - section
## - chunk text
## - metadata

## Later this can be passed to Qdrant, Ollama embeddings, or another retrieval backend.

In [31]:
final_chunks = []

for doc in normalized_docs:
    chunks = chunk_structured(doc)
    for i, ch in enumerate(chunks):
        final_chunks.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{i}",
            "doc_id": doc["doc_id"],
            "source_file": doc["source_file"],
            "section": ch["section"],
            "text": ch["text"],
            "char_len": len(ch["text"])
        })

final_chunks_df = pd.DataFrame(final_chunks)
display(final_chunks_df.head(20))
print(f"Total final chunks: {len(final_chunks_df)}")

,chunk_id,doc_id,source_file,section,text,char_len
0,2410.21169v4_chunk_0,2410.21169v4,2410.21169v4.pdf,Front Matter,5 2 0 2\nr p A 6 1\n],19
1,2410.21169v4_chunk_1,2410.21169v4,2410.21169v4.pdf,. s c [,"4 v 9 6 1 1 2 . 0 1 4 2 : v i X r a\nDocument Parsing Unveiled: Techniques, Challenges, and Prospects for Structured Data Extraction QINTONG ZHANG∗∗, Peking University, China BIN WANG∗, Shanghai A...",1692
2,2410.21169v4_chunk_2,2410.21169v4,2410.21169v4.pdf,CCS Concepts: • Computing methodologies → Natural language processing; Computer vision.,"Additional Key Words and Phrases: Document Parsing, Document OCR, Document Layout Analysis, Vision-language Model",113
3,2410.21169v4_chunk_3,2410.21169v4,2410.21169v4.pdf,ACM Reference Format:,"Qintong Zhang, Bin Wang, Victor Shea-Jay Huang, Junyuan Zhang, Zhengren Wang, Hao Liang, Conghui He, and Wentao Zhang.\n2024. Document Parsing Unveiled: Techniques, Challenges, and Prospects for S...",247
4,2410.21169v4_chunk_4,2410.21169v4,2410.21169v4.pdf,42 pages. https://doi.org/XXXXXXX.XXXXXXX,∗Authors contributed equally to this research.The work is done during an internship at Shanghai Artifcial Intelligence Laboratory. †Conghui He is the corresponding author.\nAuthors’ addresses: Qin...,1514
5,2410.21169v4_chunk_5,2410.21169v4,2410.21169v4.pdf,1 INTRODUCTION,"As digital transformation accelerates, electronic documents have increasingly replaced paper as the primary medium\nfor information exchange across various industries. This shift has broadened the...",1112
6,2410.21169v4_chunk_6,2410.21169v4,2410.21169v4.pdf,integration into modern workflows [68].,"Document parsing is crucial for document-related tasks, reshaping how information is stored, shared, and applied\nacross numerous applications. It underpins various downstream processes, including...",2798
7,2410.21169v4_chunk_7,2410.21169v4,2410.21169v4.pdf,Based on Vision Feature,"DiT [115] [137],DocGCN [151],GLAM [242]) BERTGrid [51], [41],DocLayout-YOLO [303]",81
8,2410.21169v4_chunk_8,2410.21169v4,2410.21169v4.pdf,Integrate with Semantics,"LayoutLM [268],LayoutLMv2 [269],LayoutLMv3 [89] VSR[289],Unidoc [70], [255], [184],LayoutLLM [149]",98
9,2410.21169v4_chunk_9,2410.21169v4,2410.21169v4.pdf,Text Detection,"Textboxes [126, 127], CTPN [229], DRRG [292], [312], DeepText [309], [42] [96], [155], [273], [140], [48, 262] PAN [249], CRAFT [12], SPCNET [263] LSAE [230], [114], EAST [310] CentripetalText [20...",226


Total final chunks: 322


In [32]:
## 10. Final observations

In [33]:
summary_notes = """
Preliminary notebook conclusion:

1. PyMuPDF serves as a fast flat-text baseline.
2. Unstructured provides richer semantic elements for academic PDFs.
3. Structure-aware chunking is better aligned with RefAIned than random fixed chunks.
4. The next step is to add embeddings + retrieval evaluation on these structured chunks.
"""

print(summary_notes)


Preliminary notebook conclusion:

1. PyMuPDF serves as a fast flat-text baseline.
2. Unstructured provides richer semantic elements for academic PDFs.
3. Structure-aware chunking is better aligned with RefAIned than random fixed chunks.
4. The next step is to add embeddings + retrieval evaluation on these structured chunks.



In [34]:
## Good next edit after this works

## Once this notebook runs, the first improvement I would make is changing:

## strategy="fast"

## to a comparison between:

## strategy="fast"
## strategy="hi_res"

## for the same PDFs, because that gives you a stronger experiment for class discussion.